# Dump 기반 디버깅 예시

`memory.vmem` (Linux 6.5.0-41-generic, user: `woreilly`) 실제 분석 결과를 기반으로 한
**%%ai 노트북 셀 예시 5개 + Chat UI 질문 예시 5개** 모음입니다.

## 사용 방법

1. `./scripts/start_jupyter.sh` 로 JupyterLab 시작 (API 키 자동 로드)
2. 셀 2 실행 → `%%ai` 매직 등록
3. 셀 3 실행 → 분석 결과 변수 정의
4. Part A (셀 5~9): %%ai 예시를 순서대로 실행
5. Part B (셀 11): Chat UI에 질문 텍스트 복사-붙여넣기

> `memory.vmem` 직접 로드 없이 실행 가능합니다. 변수 `network_info`, `account_info`, `context_summary`에 실제 분석 결과가 미리 하드코딩되어 있습니다.

In [ ]:
from IPython.core.magic import register_cell_magic
from litellm import completion
from IPython.display import display, Markdown
import os

@register_cell_magic
def ai(line, cell):
    """%%ai <openrouter/model> [-f markdown|code|text]"""
    parts = line.strip().split()
    model = parts[0] if parts else 'openrouter/nvidia/nemotron-3-super-120b-a12b'
    fmt = parts[parts.index('-f') + 1] if '-f' in parts else 'markdown'
    try:
        cell = cell.format_map(get_ipython().user_ns)
    except (KeyError, ValueError):
        pass
    resp = completion(
        model=model,
        messages=[{'role': 'user', 'content': cell}],
        api_key=os.environ.get('OPENROUTER_API_KEY'),
    )
    text = resp.choices[0].message.content
    display(Markdown(text) if fmt in ('markdown', 'md') else text)

NEMO = 'openrouter/nvidia/nemotron-3-super-120b-a12b'
print('%%ai 매직 등록 완료 (LiteLLM → OpenRouter, 변수 보간 지원)')

In [ ]:
import json

# 실제 memory.vmem 분석 결과를 하드코딩 (로드 시간 우회용)
network_info = {
    "external_ip_count": 17,
    "internal_ip_count": 8,
    "interesting_ports": ["20 (FTP-data)", "22 (SSH)", "23 (Telnet)", "25 (SMTP)", "53 (DNS)"],
    "note": "Telnet/FTP-data는 평문 프로토콜로 보안 위험"
}

account_info = {
    "user": "woreilly",
    "external_contacts": 17,
    "kernel_errors_present": True,
    "crash_signals": ["kernel_oops:BUG:", "crashes:crash", "alloc magic is broken"]
}

context_summary = json.dumps({
    "file": "memory.vmem",
    "kernel": "Linux 6.5.0-41-generic",
    "user": "woreilly",
    "panic_signals": [
        "alloc magic is broken at %p: %lx",
        "kernel_oops:BUG:",
        "crashes:crash"
    ],
    "memory_patterns": ["out of memory", "grub_malloc", "grub_realloc", "grub_calloc"],
    "network": network_info,
    "account": account_info
}, ensure_ascii=False, indent=2)

print('변수 준비 완료: network_info / account_info / context_summary')
print(f'context_summary 길이: {len(context_summary)} chars')

---
## Part A — `%%ai` 노트북 셀 예시 (5개)

각 셀을 실행하면 OpenRouter(nemotron) 모델이 응답합니다. (~5~10초 소요)

In [ ]:
%%ai openrouter/nvidia/nemotron-3-super-120b-a12b -f markdown
커널 메모리 덤프에서 'alloc magic is broken at %p: %lx' 문자열과
'kernel_oops:BUG:' 패턴이 동시에 발견됐어.
Linux 6.5.0-41-generic 환경에서 이 두 신호가 함께 나타날 때
가장 가능성 높은 root cause와 확인해야 할 커널 함수를 알려줘.
표 형식으로 우선순위 순으로 정리해줘.

In [ ]:
%%ai openrouter/nvidia/nemotron-3-super-120b-a12b -f markdown
메모리 덤프에서 아래 패턴들이 동시에 검출됐어:
- 'out of memory' 문자열
- grub_malloc / grub_realloc / grub_calloc 함수 패턴
- crashes:crash 신호

OOM killer가 개입했을 가능성과, grub 관련 할당자가 커널 패닉과 연결되는
시나리오를 단계별로 설명해줘. 각 단계에서 확인할 수 있는 커널 로그 패턴도 알려줘.

In [ ]:
%%ai openrouter/nvidia/nemotron-3-super-120b-a12b -f markdown
아래 네트워크 정보를 분석해줘. 외부 IP와의 통신 패턴, 활성 포트를 보고
공격자 침투 또는 데이터 유출 가능성을 평가해줘.
특히 Telnet(23)과 FTP-data(20) 포트가 동시에 활성화된 위험도와
공격자가 이를 악용하는 구체적인 시나리오를 알려줘.

{network_info}

In [ ]:
%%ai openrouter/nvidia/nemotron-3-super-120b-a12b -f markdown
아래는 Linux 시스템의 계정 및 에러 정보야.
이 계정이 공격자 계정이거나 탈취됐을 가능성을 평가하고,
메모리 덤프에서 추가로 확인할 수 있는 아티팩트의 확인 방법을 알려줘.
(bash 히스토리 잔재, 열린 파일 디스크립터, 소켓 상태 등)

{account_info}

In [ ]:
%%ai openrouter/nvidia/nemotron-3-super-120b-a12b -f markdown
아래는 Linux 메모리 덤프 전체 분석 요약이야.
발견된 모든 신호를 검토하고 P0(즉시)/P1(24h 내)/P2(이번 주 내) 우선순위로 분류해줘.
각 항목마다 다음 조사를 위한 구체적인 명령어 또는 확인 방법도 포함해줘.

{context_summary}

---
## Part B — Chat UI (`@Jupyternaut`) 질문 예시 (5개)

JupyterLab 우측 Chat UI를 열고 (`File > New > Chat`) 아래 텍스트를 붙여넣어 사용하세요.

- `@Jupyternaut`는 Jupyternaut AI 에이전트에게 직접 전달됩니다
- `@file:` 접두사로 노트북 파일을 첨부할 수 있습니다
- bash/filesystem 도구를 통해 파일 직접 읽기도 가능합니다

**질문 6. bash 도구 가용성 확인**
```
@Jupyternaut bash 도구로 /home/taejin/Jupyter/data/memory/memory.vmem 파일의 크기와 MD5 해시를 확인해줘.
```

---

**질문 7. 이 노트북 첨부 분석**
```
@Jupyternaut @file:notebooks/dump_debug_examples.ipynb 이 노트북의 %%ai 결과 중 가장 위험한 신호가 뭔지 요약해줘.
```

---

**질문 8. Telnet/FTP 공격 시나리오 심층 분석**
```
@Jupyternaut 이 덤프에서 Telnet(포트 23)과 FTP-data(포트 20)가 동시에 활성 상태였고 외부 IP 17개와 통신 흔적이 있어. Linux 6.5.0 서버에서 이 두 포트가 열려있을 때 공격자가 취할 수 있는 구체적인 공격 시나리오와 추가로 확인해야 할 프로세스/로그 위치를 알려줘.
```

---

**질문 9. woreilly 계정 후속 조사**
```
@Jupyternaut 덤프에서 유일한 사용자 계정인 woreilly가 외부 IP 17개와 통신했고 kernel_oops도 발생했어. 메모리 덤프에서 bash 히스토리 잔재, 열린 파일 디스크립터, 소켓 상태를 확인하려면 어떤 Volatility 플러그인이나 문자열 검색 패턴을 써야 해?
```

---

**질문 10. OOM killer 가설 검증 명령 생성**
```
@Jupyternaut 덤프에서 'out of memory' 신호와 grub 할당자 손상이 동시에 발견됐어. OOM killer 트리거를 검증하기 위해 memory.vmem에서 검색해야 할 커널 심볼 목록과 strings 명령 패턴을 구체적으로 알려줘.
```